## 8. CNN 网络的构建（PyTorch）

#### 1. 目的

##### 1.1 前面已经学了什么
到目前为止，我们其实已经把 CNN 的核心组件都拆开学过了：
* 卷积层 Conv2d
* 激活函数 ReLU
* 池化层 MaxPool2d / AvgPool2d
* GAP（Global Average Pooling）
* Flatten
* 全连接层 Linear

所以现在要做的事情，不再是单独认识某一个层，

而是要学会：

如何把这些模块在 PyTorch 中真正组装成一个完整的 CNN。 🧩

##### 1.2 这一节的重点
这一节我们重点学习的是：
* 在 PyTorch 中如何写一个基础 CNN
* 常规写法是什么
* 不同结构风格怎么写
* stride > 1 且不使用池化层时怎么写
* 使用 GAP 且不使用 Flatten 时怎么写

注意，这一节的重点是：

CNN 主体结构怎么搭。

至于全连接层本身的详细原理，我们之前已经学过了，

这里我只会给一个简单示例，不再重复展开。

#### 2. 在 PyTorch 中构建 CNN 的基本思路

##### 2.1 CNN 本质上就是按顺序堆层
在 PyTorch 中，一个 CNN 网络本质上就是把一层一层模块按顺序组合起来。

例如一个最基础的结构可能就是：

`Conv → ReLU → Pool → Conv → ReLU → Pool → Flatten → Linear`

所以从代码角度看，CNN 的构建本质就是两件事：
* 在 `__init__()` 中定义网络层
* 在 `forward()` 中规定数据流动顺序

##### 2.2 PyTorch 中最常见的两种写法
在 PyTorch 中写 CNN，最常见有两种方式：

**方式一：逐层定义**

就是在 __init__() 中分别定义：
* self.conv1
* self.relu
* self.pool
* self.conv2
* …

然后在 forward() 中一层一层调用。

这种方式的优点是：
* 结构清晰
* 灵活度高
* 适合初学和调试

---

**方式二：使用 nn.Sequential**
把若干层直接按顺序包起来，例如：
``` python
self.features = nn.Sequential(
    nn.Conv2d(...),
    nn.ReLU(),
    nn.MaxPool2d(...),
    ...
)
```
这种方式的优点是：
* 写法简洁
* 适合简单顺序结构

不过如果结构比较复杂，比如有分支、跳连、多个输入输出，

那么还是逐层定义更灵活。

##### 2.3 一个构建 CNN 的固定思维顺序
1. 输入图像大小是多少
2. 第一层卷积输出多少通道
3. 是否加激活函数
4. 使用池化还是使用 stride > 1 下采样
5. 第二层卷积怎么接
6. 最后是 Flatten + FC 还是 GAP + FC

##### 3. PyTorch 中 CNN 的最基础写法

##### 3.1 标准结构：Conv + ReLU + Pool
下面先看一个最基础、最经典的 CNN 示例。

假设输入是 MNIST 灰度图：

`1 × 28 × 28`

我们构建一个简单网络：
* 卷积
* ReLU
* 最大池化
* 再卷积
* 再池化
* 最后简单接一个全连接层

##### 3.2 代码示例

In [1]:
import torch 
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # 第一层卷积层
        self.conv1 = nn.Conv2d(
            in_channels=1, # 1 * 28 * 28 灰度图
            out_channels=16, # 表示16个卷积核，输出16通道特征图
            kernel_size=3, # 卷积核大小为3*3
            stride=1, # 步长为1
            padding=1 # 填充为1，保持输入输出尺寸不变
        )
        # 此时输出特征图尺寸为16 * 28 * 28
        # 激活函数
        self.relu1 = nn.ReLU()
        # 池化层
        self.pool1 = nn.MaxPool2d(
            kernel_size=2, # 池化核大小为2*2
            stride=2 # 步长为2
        )
        # 此时输出特征图尺寸为16 * 14 * 14

        # 第二层卷积层
        self.conv2 = nn.Conv2d(
            in_channels=16, # 上一层的输出通道数为16
            out_channels=32, # 表示32个卷积核，输出32通道
            kernel_size=3, # 卷积核大小为3*3
            stride=1, # 步长为1
            padding=1 # 填充为1，保持输入输出尺寸不变
        )
        # 此时输出特征图尺寸为32 * 14 * 14
        # 激活函数
        self.relu2 = nn.ReLU()
        # 池化层
        self.pool2 = nn.MaxPool2d(
            kernel_size=2, # 池化核大小为2*2
            stride=2 # 步长为2
        )
        # 此时输出特征图尺寸为32 * 7 * 7

        # flatten层，将多维特征图展平为一维向量
        self.flat = nn.Flatten()

        # 全连接层
        self.fc1 = nn.Linear(
            in_features=32*7*7, # 上一层输出的特征图展平后的维度
            out_features=num_classes # 输出类别数
        )

    def forward(self, x):
        x = self.conv1(x) # 卷积层1  [B, 1, 28, 28] -> [B, 16, 28, 28]
        x = self.relu1(x) # 激活函数1 [B, 16, 28, 28] -> [B, 16, 28, 28]
        x = self.pool1(x) # 池化层1 [B, 16, 28, 28] -> [B, 16, 14, 14]

        x = self.conv2(x) # 卷积层2 [B, 16, 14, 14] -> [B, 32, 14, 14]
        x = self.relu2(x) # 激活函数2 [B, 32, 14, 14] -> [B, 32, 14, 14]
        x = self.pool2(x) # 池化层2 [B, 32, 14, 14] -> [B, 32, 7, 7]

        x = self.flat(x) # flatten层   [B, 32, 7, 7] -> [B, 32*7*7]

        x = self.fc1(x) # 全连接层 [B, 32*7*7] -> [B, num_classes]

        return x

##### 3.3 这个结构怎么理解
这个网络的流程就是：
* 第一层卷积提取浅层特征
    * 16个 kernel，所以out_channels=16
    * 得到的特征图为28 * 28 * 16
* 第一层池化压缩尺寸
    * MaxPooling 之后最终特征图为 14 * 14 * 16
* 第二层卷积提取更高层特征
    * 上一层特征图通道数量为 16，所以 in_channels=16
    * 32个Kernal, 所以 out_channels=32
    * 得到的特征图为 14 * 14 *32 
* 第二层池化进一步压缩尺寸
    * MaxPooling之后的最终特征图为 7 * 7 * 32
* Flatten 展平
    * 此时特征图为 7 * 7 * 32
* 全连接层输出分类结果
    * 所以全连接层的 Linear 输入神经元个数为 32 * 7 * 7
这就是最标准的入门 CNN 写法。

#### 4. 用 nn.Sequential 写基础 CNN

##### 4.1 什么时候适合用 Sequential
如果你的网络结构是“严格按顺序一路往下走”的，

那么可以直接用 nn.Sequential 简化写法。

对于基础 CNN，这种写法非常常见。

##### 4.2 代码示例

In [2]:
class SimpleSequentialCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # 特征提取部分， 卷积层 + 激活函数 + 池化层
        self.feature_extractor = nn.Sequential(
            # 第一层卷积层
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1), # [B, 1, 28, 28] -> [B, 16, 28, 28]
            # 激活函数
            nn.ReLU(), # [B, 16, 28, 28] -> [B, 16, 28, 28]
            # 池化层
            nn.MaxPool2d(kernel_size=2, stride=2), # [B, 16, 28, 28] -> [B, 16, 14, 14]

            # 第二层卷积层
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), # [B, 16, 14, 14] -> [B, 32, 14, 14]
            # 激活函数
            nn.ReLU(), # [B, 32, 14, 14] -> [B, 32, 14, 14]
            # 池化层
            nn.MaxPool2d(kernel_size=2, stride=2) # [B, 32, 14, 14] -> [B, 32, 7, 7]
        )
        # 分类器部分， flatten层 + 全连接层
        self.classifier = nn.Sequential(
            nn.Flatten(), # [B, 32, 7, 7] -> [B, 32*7*7]
            nn.Linear(32*7*7, num_classes) # [B, 32*7*7] -> [B, num_classes]
        )

    def forward(self, x):
        x = self.feature_extractor(x) # 特征提取部分
        x = self.classifier(x) # 分类器部分
        return x

#### 5. CNN 中常见的“模块化”构建方式

##### 5.1 为什么要模块化
当网络稍微复杂一点后，我们通常不会每次都手动写：
* Conv
* ReLU
* Pool

而是会把常见组合封装起来。

例如，一个很常见的小模块就是：

`Conv + ReLU`

或者：

`Conv + BatchNorm + ReLU`

##### 5.2 构建模块（卷积层 + 激活函数）

In [4]:
class SimpleConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding
        )
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        return x

##### 5.3 在主网络中复用
这种写法在真实项目中会更常见，因为结构更清晰。

In [5]:
class SimpleCNNWithBlocks(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # 第一层卷积块
        self.conv1 = SimpleConvBlock(1, 16, 3, 1, 1) 
        # 第一层池化层
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # 第二层卷积块
        self.conv2 = SimpleConvBlock(16, 32, 3, 1, 1)
        # 第二层池化层
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # flatten层，将多维特征图展平为一维向量
        self.flat = nn.Flatten()
        # 全连接层
        self.fc1 = nn.Linear(32*7*7, num_classes)

    def forward(self, x):
        x = self.conv1(x) # 卷积块1
        x = self.pool1(x) # 池化层1
        x = self.conv2(x) # 卷积块2
        x = self.pool2(x) # 池化层2
        x = self.flat(x) # flatten层
        x = self.fc1(x) # 全连接层
        return x

#### 6. stride > 1 且不使用池化层的 CNN 写法

##### 6.1 为什么可以不用池化层
我们前面已经学过，在现代网络中：

下采样不一定非要通过池化层来完成。

如果卷积层本身设置：

`stride = 2`

那么它在提取特征的同时，也能把空间尺寸缩小。

也就是说，它可以同时完成：
* 特征提取
* 下采样

##### 6.2 这种结构的核心思想
普通写法是：

`Conv → ReLU → Pool`

而这种写法变成：

`Conv(stride=2) → ReLU`

这样就把池化层省掉了。

##### 6.3 代码示例

In [ ]:
class StridedCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv_1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1) # [B, 1, 28, 28] -> [B, 16, 14, 14]
        self.relu_1 = nn.ReLU()
        self.conv_2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1) # [B, 16, 14, 14] -> [B, 32, 7, 7]
        self.relu_2 = nn.ReLU()

        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(32*7*7, num_classes)

    def forward(self, x):
        x = self.conv_1(x)
        x = self.relu_1(x)
        x = self.conv_2(x)
        x = self.relu_2(x)
        x = self.flat(x)
        x = self.fc1(x)
        return x

##### 6.4 输入输出形状变化
这里没有池化层，但空间尺寸仍然变化了：
* 28 × 28 → 14 × 14
* 14 × 14 → 7 × 7

原因就是卷积层用了 stride=2。

#### 7. 使用 GAP 代替 Flatten 的 CNN 写法

##### 7.1 为什么 GAP 可以替代 Flatten
我们前面学过：
`Global Average Pooling（GAP）` 会把每个通道的整张特征图压缩成一个值。

例如：
* 输入：[B, 32, 7, 7]
* GAP 后：[B, 32, 1, 1]

如果再把这两个长度为 1 的空间维去掉，

那么就变成：

`[B, 32]`

这时就已经可以直接送入一个线性层了。

所以这种结构里通常：
* 不需要传统的 Flatten(H×W×C)
* 只需要把 1×1 压掉即可

##### 7.2 这种结构的思路
这种写法通常是：

`Conv → ReLU → Conv → ReLU → GAP → Linear`

它和传统：

`Conv → ReLU → Pool → Flatten → FC`

相比，更现代、更轻量。

##### 7.3 代码示例

In [7]:
class GAPCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv_1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1) # [B, 1, 28, 28] -> [B, 16, 28, 28]
        self.relu_1 = nn.ReLU()
        self.pool_1 = nn.MaxPool2d(kernel_size=2, stride=2) # [B, 16, 28, 28] -> [B, 16, 14, 14]
        self.conv_2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1) # [B, 16, 14, 14] -> [B, 32, 14, 14]
        self.relu_2 = nn.ReLU()
        self.pool_2 = nn.MaxPool2d(kernel_size=2, stride=2) # [B, 32, 14, 14] -> [B, 32, 7, 7]

        self.gap = nn.AdaptiveAvgPool2d((1, 1)) # 全局平均池化层 [B, 32, 7, 7] -> [B, 32, 1, 1]
        self.fc = nn.Linear(32, num_classes) # 全连接层 [B, 32] -> [B, num_classes]

    def forward(self, x):
        x = self.conv_1(x)
        x = self.relu_1(x)
        x = self.pool_1(x)
        x = self.conv_2(x)
        x = self.relu_2(x)
        x = self.pool_2(x)
        x = self.gap(x) # [B, 32, 7, 7] -> [B, 32, 1, 1]
        x = torch.flatten(x, 1) # [B, 32, 1, 1] -> [B, 32]
        x = self.fc(x) # [B, 32] -> [B, num_classes]
        return x

##### 7.4 这里为什么还用了 torch.flatten(x, 1)⚠️
这里要特别说明一下：

严格来说，这里不是传统意义上的 Flatten 整个特征图，

因为 GAP 之后空间已经变成了 `1 × 1`。

此时 `torch.flatten(x, 1)` 的作用更像是：

去掉多余的 `1 × 1` 空间维度，把 `[B, C, 1, 1]` 变成 `[B, C]`。

所以这种结构通常被理解为：

GAP 代替了传统的 “大 Flatten”。

#### 8. 三种 CNN 构建风格的整体对比

##### 8.1 常规写法：卷积 + 池化
结构：

`Conv → ReLU → Pool → Conv → ReLU → Pool → Flatten → FC`

特点：
* 最经典
* 最适合教学和入门
* 结构清晰

##### 8.2 stride > 1 无池化写法
结构：

`Conv(stride=2) → ReLU → Conv(stride=2) → ReLU → Flatten → FC`

特点：
* 不单独使用池化层
* 由卷积直接完成下采样
* 更偏现代网络思路

##### 8.3 GAP 无大 Flatten 写法
结构：
Conv → ReLU → … → GAP → Linear
特点：
* 参数更少
* 网络更轻
* 最后阶段很常见